In [1]:
import logging
from dataclasses import asdict
import sys
from pathlib import Path

from dotenv import load_dotenv
import pandas as pd
from datasets import load_dataset
from IPython.display import display

logging.basicConfig(level=logging.INFO)

In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "SWE-Gym/OpenHands-Sampled-Trajectories",
    split="train.raw",
)

df = dataset.to_pandas().merge(
    load_dataset("SWE-Gym/SWE-Gym", split="train").to_pandas(),
    on="instance_id",
    how="left",
)

print(f"Trajectories: {len(dataset):,} | Combined rows: {len(df):,}")
print(df.columns.tolist())

Trajectories: 6,055 | Combined rows: 6,055
['instance_id', 'run_id', 'resolved', 'messages', 'tools', 'test_result', 'hints_text', 'patch', 'test_patch', 'created_at', 'problem_statement', 'repo', 'base_commit', 'version', 'PASS_TO_PASS', 'FAIL_TO_PASS']


### 1.b Inspect messages in order

Each row is a recorded message. `tool_call_id` links a tool response to its call when present.

In [3]:
sample_index = 100
sample = df.iloc[sample_index]
print(f"[STAGE] Inspect record {sample_index}")
print("✓ Fields:", df.columns)
display(pd.Series({key: sample.get(key) for key in ["instance_id", "run_id", "resolved", "test_result"]}, name="value").to_frame())
messages = sample.get("messages")
tools = sample.get("tools")
print("Messages:", len(messages) if messages is not None else 0)
print("Available tools:", [tool.get("function", {}).get("name") for tool in (tools if tools is not None else [])])

[STAGE] Inspect record 100
✓ Fields: Index(['instance_id', 'run_id', 'resolved', 'messages', 'tools', 'test_result',
       'hints_text', 'patch', 'test_patch', 'created_at', 'problem_statement',
       'repo', 'base_commit', 'version', 'PASS_TO_PASS', 'FAIL_TO_PASS'],
      dtype='object')


,value
instance_id,python__mypy-10308
run_id,gpt-4o-2024-08-06_maxiter_30_N_v2.1-no-hint-tr...
resolved,True
test_result,{'apply_patch_output': 'Checking patch mypy/co...


Messages: 27
Available tools: ['execute_bash', 'finish', 'str_replace_editor']


In [5]:
import json

message_df = pd.DataFrame(list(messages) if messages is not None else [])
message_df["message_index"] = message_df.index
message_df = message_df.explode("tool_calls", ignore_index=False)
message_df.tail()

,content,function_call,name,role,tool_call_id,tool_calls,message_index
22,The indentation issue in `constraints.py` has ...,None,None,assistant,None,"{'function': {'arguments': '{""command"":""mypy -...",22
23,OBSERVATION:\nmypy --follow-imports=silent --s...,None,execute_bash,tool,call_3z4aXWUYgT50BeZ94XOK0RnI,None,23
24,The `mypy` check on the `reproduce_error.py` s...,None,None,assistant,None,None,24
25,Please continue working on the task on whateve...,None,None,user,None,None,25
26,The issue described in the PR has been success...,None,None,assistant,None,"{'function': {'arguments': '{}', 'name': 'fini...",26


### Verifying Outcome

In [6]:
call_details = pd.json_normalize([
    call if isinstance(call, dict) else {}
    for call in message_df.pop("tool_calls")
]).rename(columns={
    "id": "call_id",
    "type": "call_type",
    "function.name": "call_name",
    "function.arguments": "call_arguments",
})
call_details

,call_id,index,call_type,call_arguments,call_name
0,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN
2,call_47R2euoxoLgwG3lxgPTg6GzV,NaN,function,"{""command"":""view"",""path"":""/workspace/python__m...",str_replace_editor
3,NaN,NaN,NaN,NaN,NaN
4,call_FvAWNqGgXEaat0c3CJ4spD4z,NaN,function,"{""command"":""view"",""path"":""/workspace/python__m...",str_replace_editor
5,NaN,NaN,NaN,NaN,NaN
6,call_0Lrhg51CWkBAmnp3ysU5w43w,NaN,function,"{""command"":""create"",""path"":""/workspace/python_...",str_replace_editor
7,NaN,NaN,NaN,NaN,NaN
8,call_09Q9FZHPtif45IstaeuI3YAb,NaN,function,"{""command"":""python3 /workspace/python__mypy__0...",execute_bash
9,NaN,NaN,NaN,NaN,NaN
